# S6 · Combinations: feedback sources × optimizers

Study path stage 6 of 7 · Chapter 13 · Day 23 · **DGX Spark GPU first**

Stages S1–S5 studied each piece on its own: a reward model (S1), the KL-regularized optimum and DPO
(S2), policy gradients and PPO (S3), GRPO (S4) and verifier rewards (S5). Real post-training
pipelines combine them. In D40 we separated two axes: the **feedback source** (human pairs, a learned
reward model, an LLM judge, a verifier) and the **optimizer** (selection + SFT, the DPO loss, PPO or
GRPO). This notebook builds the combinations used in practice and measures what each one converges
to. When you finish, you can wire any feedback source to any update rule, predict the result on a
small exact model, and run one round of reward-model-labelled online DPO on Qwen3-0.6B.

| Combination | Feedback source | Update rule | Example |
|---|---|---|---|
| best-of-n, rejection-sampling fine-tuning | reward model | select, then SFT on the selections | Llama 2 (before PPO), Llama 3 |
| online / iterative DPO | reward model or LLM judge labels **new** pairs | DPO loss | Self-Rewarding LMs (Yuan et al., 2024) |
| RM + verifier reward | both, mixed into one number per answer | policy gradient (GRPO) | DeepSeek-R1's last RL stage instead routes per prompt: rules for reasoning, an RM for general prompts |
| SFT → DPO → RLVR | demonstrations → preference pairs → verifier | NLL → DPO → policy gradient (PPO) | Tülu 3 |

| Part | Question | Discussions in the [learning note](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md) |
|---|---|---|
| A · Best-of-n and rejection sampling | What does selecting with a reward model do to the answer distribution, and what does SFT on the selections converge to? | D15, D32, D40 |
| B · Offline vs online DPO | DPO has no reward model inside. What changes when a reward model labels pairs that the current policy samples? | D14, D34, D35, D40 |
| C · Reward model + verifier | How do you mix a learned score with a 0/1 verifier so that a wrong answer can't win? | D32, D33 |
| D · SFT → DPO → RLVR | What does each stage of a Tülu 3-style pipeline change, and what keeps a later stage from undoing an earlier one? | D33, D34, D35 |
| E · Qwen3: reward-model-labelled online DPO | After one round, does the labelling reward model score fresh answers higher, and does an independently trained reward model agree? | D15, D34, D40 |

RL terms (policy, reward, return, advantage, baseline, on-policy) are defined in the glossary
section **RL terms in the LLM setting** of the [learning note](../../learning_artifacts/day-15-preference-data-and-bradley-terry/learning-from-preferences.md).

## How the exercises work

Each exercise asks you to **write the key lines**: usually one to three, never a large block.

1. **Read the task**, then write your lines where you see `...` in the exercise cell.
2. **Run the check.** It tests your code on small cases and prints ✅ or ❌ with a hint. Nothing
   crashes if you haven't finished; you'll see ⏳ instead.
3. **Compare with the reference**, which follows each check and then uses the function in a small
   experiment.

Later cells use your version when its check passed, and the reference otherwise
(`mine_or_ref(globals(), …)`), so one stuck exercise never blocks the rest. Predict cells ask for
a written answer first; the reference computes it.

**Data.** Parts A–D use small hand-written tables (one or two prompts, six answers each), so every
number can be computed exactly. Part E uses [Anthropic HH-RLHF](https://huggingface.co/datasets/Anthropic/hh-rlhf)
`helpful-base` (MIT license): two disjoint sets of 2,000 training pairs for two reward models,
single-turn training questions from `train[20000:]` (never seen by either reward model) as online
DPO prompts, and test questions and pairs for evaluation.

**Environment.** Kernel `Python (DGX Spark Native)` with CUDA; Qwen3-0.6B weights and the HH-RLHF
files in the local Hugging Face cache. Nothing is downloaded. Checks and reference implementations
come from the tested module `dongxi_llms.study_path.s6_combinations`.

**Run time.** About 1 minute for Parts A–D, about 11 minutes for Part E on the Spark GPU (most of it training the two reward models).

**Scale and limits.** Parts A–D are exact or near-exact statements about tabular models: they show
mechanisms (what selection, pair source, reward mixing and KL anchors do), not how large the
effects are in a network that shares parameters across prompts. Part E is one short online round
on a 0.6B model with reward models trained on 2,000 pairs each (about 55% held-out accuracy). It
verifies the mechanics and measures the change in reward-model score with standard errors; it
makes no claim that answers became more helpful.

In [ ]:
import inspect, math, random, sys, time
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/dongxi_llms").is_dir())
sys.path.insert(0, str(ROOT / "src"))               # the course package, wherever this copy lives
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, torch.nn.functional as F
from dongxi_llms.study_path import qwen as Q
from dongxi_llms.study_path import s6_combinations as s6
from dongxi_llms.study_path.common import mine_or_ref

assert torch.cuda.is_available(), "Use the Python (DGX Spark Native) kernel: this notebook is GPU first."
DEVICE = "cuda"
random.seed(0); np.random.seed(0); torch.manual_seed(0)
torch.set_default_dtype(torch.float64)       # the toy tables (Parts A–D) are exact; Qwen3 sets its own dtypes
plt.rcParams.update({"figure.facecolor": "white", "axes.facecolor": "white", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})
BLUE, ORANGE, GRAY, GREEN, RED = "#2563EB", "#EA580C", "#94A3B8", "#16A34A", "#DC2626"
T_START = time.time()
print(torch.cuda.get_device_name(0), "| torch", torch.__version__)

---

# Part A · Best-of-n and rejection-sampling fine-tuning

A reward model can improve answers without changing any weights. Sample `n` answers to a prompt,
score each with the reward model, and keep the highest-scored one. This is **best-of-n selection**
(BoN). It spends inference compute instead of training compute.

If we then fine-tune the policy on the kept answers, the selection becomes part of the weights. This
is **rejection-sampling fine-tuning** (RSFT): generate, select with the reward model, run SFT on the
selections. Llama 2-Chat used it for its first RLHF rounds and then ran PPO on top. The feedback
source is the reward model; the update rule is plain supervised learning.

### The toy world used in Parts A–C

One prompt, *Can dogs eat raw garlic?* (the glossary's running example), and six possible answers.
The **policy** is a table: one probability per answer. Each answer has a **true reward** `r*`, the
quality we actually want (unknown in real training), and a **reward-model score** `r̂`, a learned
proxy for it. The proxy ranks most answers correctly, but it over-rates `y4`, a fluent, enthusiastic
answer that is wrong. A **verifier** (Part C) checks one fact: the answer says *no* and gives the
reason that garlic is toxic.

In [ ]:
answers = ["No. Garlic is toxic to dogs; it damages red blood cells.",   # y0
           "No, it's toxic.",                                           # y1
           "Small amounts are fine.",                                   # y2
           "I'm not sure. Ask a vet.",                                  # y3
           "Great question! Garlic has many health benefits for dogs and people alike, ...",  # y4
           "Yes."]                                                      # y5
K = len(answers)
pi_ref   = torch.tensor([0.10, 0.25, 0.20, 0.25, 0.05, 0.15])   # π_ref(y): the starting policy
r_true   = torch.tensor([2.0,  1.0, -2.0,  0.0, -1.5, -3.0])    # r*(y): what we want (hidden in practice)
r_rm     = torch.tensor([1.5,  0.5, -1.0,  0.0,  2.5, -2.0])    # r̂(y): the reward model's score
verifier = torch.tensor([1.0,  1.0,  0.0,  0.0,  0.0,  0.0])    # v(y): 1 if it says no + toxic
assert torch.isclose(pi_ref.sum(), torch.tensor(1.0))
world = pd.DataFrame({"answer": answers, "π_ref": pi_ref, "r* (true)": r_true, "r̂ (RM)": r_rm,
                      "verifier": verifier.int()}, index=[f"y{i}" for i in range(K)])
print(world.to_string())
print(f"\nE_ref[r*] = {pi_ref @ r_true:+.3f} | E_ref[r̂] = {pi_ref @ r_rm:+.3f} | "
      f"RM ranks y4 first, the true reward ranks it fifth")

### Prediction

## Exercise 1 · best-of-n selection

`samples` has shape `[B, n]`: `B` independent groups of `n` answer indices drawn from the policy.
`scores` has shape `[K]`: the reward-model score of each answer. Return, for each group, the
**answer index** (not its position in the row) with the highest score: shape `[B]`.

Two steps: score each sample with `scores[samples]`, find the position of the maximum in each row
(`argmax(1, keepdim=True)`), and map that position back to an answer with `samples.gather(1, …)`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_best_of_n(samples, scores):
    pos = ...                                   # [B, 1] position of the highest-scored sample per row
    return ...                                  # [B] the answer index at that position

In [ ]:
ex1_ok = s6.check_best_of_n(globals().get("my_best_of_n"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def best_of_n_ref(samples, scores):
    return samples.gather(1, scores[samples].argmax(1, keepdim=True)).squeeze(1)

s6.check_best_of_n(best_of_n_ref, strict=True)
best_of_n = mine_or_ref(globals(), "my_best_of_n", ex1_ok, best_of_n_ref)

g = torch.Generator().manual_seed(0)
groups = torch.multinomial(pi_ref, 100_000 * 4, replacement=True, generator=g).view(-1, 4)   # [100000, 4]
picked = best_of_n(groups, r_rm)
freq_bo4 = torch.bincount(picked, minlength=K) / len(picked)
print("π_ref          ", [f"{x:.3f}" for x in pi_ref])
print("best-of-4 freq ", [f"{x:.3f}" for x in freq_bo4])
print(f"E[r*]: {pi_ref @ r_true:+.3f} → {freq_bo4 @ r_true:+.3f} | E[r̂]: {pi_ref @ r_rm:+.3f} → {freq_bo4 @ r_rm:+.3f}")

### Reference explanation

### Reference answer and mechanism

`scores[samples]` looks up a score for every sample, shape `[B, n]`. `argmax(1)` gives the
*position* of the best sample in its row, and `gather` turns that position into the answer index.
Returning the position is the usual bug: the check catches it.

With `n = 4`, selection moves mass from the low-scored answers (`y2`, `y3`, `y5`) to the high-scored
ones. Both the reward-model score and the true reward rise, because for most answers the two agree.
`y4`'s share also rises, from 0.05 to about 0.19, because the reward model ranks it first: whenever
`y4` is in the group, it is selected.

### Prediction

## Exercise 2 · the exact best-of-n distribution

Monte Carlo needs many samples. For a finite answer set the best-of-n distribution has a closed
form. Sort the answers by score from lowest to highest and let `F_k = p_(1) + … + p_(k)` be the
probability that one draw scores at most the k-th answer. The best of `n` independent draws is the
k-th answer exactly when all `n` draws score at most the k-th answer, but not all of them score
below it:

`P(best = k-th answer) = F_k^n − F_(k−1)^n`,  with `F_(k−1) = F_k − p_(k)`.

Write that line (scores are assumed distinct). The scaffold sorts and un-sorts for you.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_bon_probs(p, scores, n):
    order = scores.argsort()                    # answers from lowest to highest score
    p_sorted = p[order]
    cdf = p_sorted.cumsum(0)                    # F_k
    p_best_sorted = ...                         # P(best of n = k-th answer), in sorted order
    out = torch.empty_like(p); out[order] = p_best_sorted
    return out

In [ ]:
ex2_ok = s6.check_bon_probs(globals().get("my_bon_probs"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s6.check_bon_probs(s6.ref_bon_probs, strict=True)
bon_probs = mine_or_ref(globals(), "my_bon_probs", ex2_ok, s6.ref_bon_probs)
print(inspect.getsource(s6.ref_bon_probs))

exact_bo4 = bon_probs(pi_ref, r_rm, 4)
print("exact best-of-4", [f"{x:.3f}" for x in exact_bo4])
print(f"max |Monte Carlo − exact| = {(freq_bo4 - exact_bo4).abs().max():.4f} (100,000 groups)")
assert (freq_bo4 - exact_bo4).abs().max() < 0.005

### Reference explanation

### Reference answer and mechanism

`cdf ** n` is the probability that all `n` draws land at or below the k-th answer, i.e. that the
best draw is *at most* the k-th. Subtracting `(cdf − p_sorted) ** n`, the probability that all draws
are strictly below it, leaves the probability that the best draw is *exactly* the k-th. At `n = 1`
this gives back `p`; as `n` grows, all mass moves to the top-scored answer.

The Monte Carlo frequencies from Exercise 1 agree with the formula to within sampling noise. From
here on we use the exact formula, so the curves below have no noise at all.

### Prediction

## Predict 1 · more samples, better answers?

Best-of-n uses the reward model `r̂`, which over-rates `y4`. As `n` grows from 1 to 256, what
happens to (a) the expected **reward-model score** of the selected answer, and (b) its expected
**true reward** `r*`? Does (b) keep rising, level off, or turn down, and roughly at which `n`?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
ns = [1, 2, 3, 4, 6, 8, 12, 16, 24, 32, 48, 64, 96, 128, 192, 256]
rows = []
for n in ns:
    pb = bon_probs(pi_ref, r_rm, n)           # select with the reward model
    po = bon_probs(pi_ref, r_true, n)         # select with the true reward (an oracle we never have)
    rows.append({"n": n, "true reward (RM selects)": float(pb @ r_true), "RM score (RM selects)": float(pb @ r_rm),
                 "true reward (oracle selects)": float(po @ r_true), "P(y4)": float(pb[4]),
                 "KL(bon ‖ π_ref)": float((pb * (pb / pi_ref).log()).nansum()),
                 "bound log n − (n−1)/n": math.log(n) - (n - 1) / n})
bon_table = pd.DataFrame(rows).set_index("n")
print(bon_table.loc[[1, 2, 4, 8, 16, 64, 256]].round(3).to_string())
n_peak = int(bon_table["true reward (RM selects)"].idxmax())
print(f"true reward peaks at n = {n_peak}: {bon_table.loc[n_peak, 'true reward (RM selects)']:+.3f}")
assert (np.diff(bon_table["RM score (RM selects)"]) > 0).all()
assert (bon_table["KL(bon ‖ π_ref)"] <= bon_table["bound log n − (n−1)/n"] + 1e-9).all()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
ax = axes[0]
ax.plot(ns, bon_table["RM score (RM selects)"], "o-", color=ORANGE, ms=3, label="RM score r̂, RM selects")
ax.plot(ns, bon_table["true reward (RM selects)"], "o-", color=BLUE, ms=3, label="true reward r*, RM selects")
ax.plot(ns, bon_table["true reward (oracle selects)"], "--", color=GREEN, label="true reward r*, oracle selects")
ax.axvline(n_peak, color=GRAY, lw=0.8, ls=":")
ax.set_xscale("log", base=2); ax.set_xlabel("n (samples per prompt)"); ax.set_ylabel("expected value of the selected answer")
ax.set_title("best-of-n with a flawed reward model"); ax.legend(fontsize=8)
ax = axes[1]
ax.plot(ns, bon_table["KL(bon ‖ π_ref)"], "o-", color=BLUE, ms=3, label="KL(best-of-n ‖ π_ref), exact")
ax.plot(ns, bon_table["bound log n − (n−1)/n"], "--", color=GRAY, label="log n − (n−1)/n")
ax.set_xscale("log", base=2); ax.set_xlabel("n"); ax.set_ylabel("nats"); ax.set_title("how far selection moves the distribution")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

(a) The **reward-model score rises monotonically**: selecting the maximum of more draws can only
raise the expected maximum. (b) The **true reward rises, peaks and then falls**. In this table it
peaks at `n = 4` (about +0.67, up from −0.48 at `n = 1`), then declines toward `r*(y4) = −1.5`,
because `P(y4 is in the group) = 1 − 0.95ⁿ` grows with `n` and the reward model always picks `y4`
when it is present. At `n = 64` the selected answer is `y4` 96% of the time. The oracle curve, which
selects with `r*` itself, keeps rising to `r*(y0) = 2`.

This is **reward-model overoptimization**: the more strongly you optimize a proxy, the more the
result reflects the proxy's errors rather than the target. Gao et al. (2022) measured the same
rise-then-fall shape for best-of-n and PPO against a larger "gold" reward model.

The right panel gives the price in KL: best-of-n moves the distribution by at most
`log n − (n−1)/n` nats (a known upper bound; the exact KL stays below it at every n here). KL grows only like `log n`, so selection is a
cheap way to move far from `π_ref`, and it has no β to stop it at a chosen distance.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Best-of-n with a flawed reward model: true reward peaks then falls; KL grows like log n](../figures/chapter-13/study-path-s6_combinations-01.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

### Prediction

## Exercise 3 · SFT on the selected answers

Rejection-sampling fine-tuning trains the policy on the selected answers with the ordinary SFT
loss: the mean negative log-likelihood. Our policy is a table of `K` logits, `π = softmax(logits)`,
and `picked` is a vector of `M` selected answer indices. Return the scalar
`−(1/M) Σ_m log π(picked_m)`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_sft_loss(logits, picked):
    return ...                                  # one line: mean NLL of the picked answers

In [ ]:
ex3_ok = s6.check_sft_loss(globals().get("my_sft_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
def sft_loss_ref(logits, picked):
    return -torch.log_softmax(logits, -1)[picked].mean()

s6.check_sft_loss(sft_loss_ref, strict=True)
sft_loss = mine_or_ref(globals(), "my_sft_loss", ex3_ok, sft_loss_ref)

def rsft_round(p_start, n, M=20_000, steps=300, seed=0):
    """One round of rejection-sampling fine-tuning: sample M groups of n, select with r̂, SFT on the selections."""
    g = torch.Generator().manual_seed(seed)
    picked = best_of_n(torch.multinomial(p_start, M * n, replacement=True, generator=g).view(M, n), r_rm)
    theta = p_start.log().clone().requires_grad_(True)       # start from the current policy
    opt = torch.optim.Adam([theta], lr=0.05)
    for _ in range(steps):
        loss = sft_loss(theta, picked)
        opt.zero_grad(); loss.backward(); opt.step()
    return torch.softmax(theta, -1).detach()

N_RS = 4
rounds = [pi_ref]
for k in range(3):
    rounds.append(rsft_round(rounds[-1], N_RS, seed=k))
for k, p in enumerate(rounds):
    exact = bon_probs(pi_ref, r_rm, N_RS ** k)
    print(f"round {k}: π = {[f'{x:.2f}' for x in p]} | E[r*] {p @ r_true:+.2f} | E[r̂] {p @ r_rm:+.2f} | "
          f"max |π − best-of-{N_RS ** k}| = {(p - exact).abs().max():.3f}")
# the exact identity behind those numbers: selecting twice with best-of-4 = selecting once with best-of-16
torch.testing.assert_close(bon_probs(bon_probs(pi_ref, r_rm, 4), r_rm, 4), bon_probs(pi_ref, r_rm, 16))
assert (rounds[1] - bon_probs(pi_ref, r_rm, 4)).abs().max() < 0.02

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2), gridspec_kw={"width_ratios": [2, 1]})
width = 0.2
for k, (p, color) in enumerate(zip(rounds, [GRAY, BLUE, ORANGE, RED])):
    axes[0].bar(np.arange(K) + (k - 1.5) * width, p, width, color=color, label=f"round {k}" + (" (π_ref)" if k == 0 else f" = best-of-{N_RS ** k}"))
axes[0].set_xticks(range(K), [f"y{i}" + ("  ✗RM" if i == 4 else "") for i in range(K)])
axes[0].set_ylabel("probability"); axes[0].set_title("policy after each round of rejection-sampling SFT (n = 4)")
axes[0].legend(fontsize=8)
axes[1].plot(range(4), [float(p @ r_true) for p in rounds], "o-", color=BLUE, label="true reward r*")
axes[1].plot(range(4), [float(p @ r_rm) for p in rounds], "o-", color=ORANGE, label="RM score r̂")
axes[1].set_xticks(range(4)); axes[1].set_xlabel("round"); axes[1].set_title("expected reward of the policy")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

`log_softmax(logits)[picked]` is `log π` of each selected answer; negate and average. Summing
instead of averaging gives the same optimum but a learning rate that depends on `M`.

The gradient of this loss with respect to the logits is `π − f`, where `f` is the empirical
frequency of the picked answers (the check verifies it). So SFT converges to `f`, and with many
samples `f` is the best-of-n distribution: **one round of RSFT turns best-of-n into the policy
itself**, with no extra inference cost. Round 1 matches best-of-4 to within about 0.01.

Repeating the round with the **same** reward model compounds the selection. Selecting the best of
4 from a policy that already is best-of-4 equals selecting the best of 16 from `π_ref` (the
`assert_close` verifies this identity exactly). After `k` rounds the policy is best-of-`4^k`: round 2
is best-of-16 (true reward already lower than round 1) and round 3 is best-of-64, 96% `y4`. The
reward-model score rises every round while the true reward falls: Part A's overoptimization, now
baked into the weights.

What stopped this in Llama 2: new human preference data was collected on the latest model's
outputs and the reward model was retrained between rounds, so errors like `y4` could be labelled and
corrected; samples from earlier rounds were also kept in the training mix. A frozen reward model
plus repeated selection has neither safeguard.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Rejection-sampling SFT: k rounds of best-of-4 equal best-of-4^k; RM score rises, true reward falls](../figures/chapter-13/study-path-s6_combinations-02.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part B · Offline vs online DPO: where do the pairs come from?

DPO (S2) is a loss on preference pairs. It has **no reward model inside**: its "reward" is the
implicit `β·log π(y)/π_ref(y)`. So "DPO with a reward model" can only mean one thing (D40): the
reward model **labels pairs**, and DPO trains on them. The new axis is **where the pairs come from**.

- **Offline DPO**: a fixed dataset of pairs, written once by other models (or people) and labelled
  once. The policy never sees its own answers.
- **Online (iterative) DPO**: at every step the *current* policy samples two answers to a prompt,
  a reward model or an LLM judge labels them, and DPO trains on that pair. Self-Rewarding LMs
  (Yuan et al., 2024) do this in rounds with the model itself as the judge.

### The target: the KL-regularized optimum

Both versions aim at the same objective (D35):

`J(π) = E_{y∼π}[r̂(y)] − β·KL(π ‖ π_ref)`.

Its maximizer is `π*(y) = π_ref(y)·exp(r̂(y)/β) / Z`, with `Z = Σ_y π_ref(y)·exp(r̂(y)/β)`. How far is
any policy from it? Substitute `r̂(y) = β·log(π*(y)/π_ref(y)) + β·log Z`:

`J(π) = Σ_y π(y)·[β·log π*(y) − β·log π_ref(y) + β·log Z − β·log π(y) + β·log π_ref(y)]`
`     = β·log Z − β·Σ_y π(y)·log(π(y)/π*(y))`
`     = J(π*) − β·KL(π ‖ π*)`.

So **`KL(π ‖ π*)` is exactly the objective gap divided by β**. We use it as the distance to the
optimum.

DPO's loss is minimized at `π*` when the labels follow the Bradley–Terry model of the reward,
`P(a ≻ b) = σ(r̂(a) − r̂(b))`, and when the pairs connect all answers. Real pipelines usually label
**deterministically** instead (chosen = higher score) and drop near-ties with a margin. Exercise 4
writes both rules.

### Prediction

## Exercise 4 · two ways for a reward model to label a pair

Each pair has scores `s_a` and `s_b` (tensors of shape `[B]`). Write two labelling rules; each
returns `(a_wins, keep)`, two boolean tensors of shape `[B]`.

- `my_label_hard(s_a, s_b, margin)`: `a` wins when `s_a > s_b`; keep the pair only when the score
  gap is at least `margin` in absolute value (near-ties are dropped as uninformative).
- `my_label_bt(s_a, s_b, generator)`: `a` wins **with probability** `σ(s_a − s_b)`, the
  Bradley–Terry probability; draw it with `torch.rand(s_a.shape, generator=generator)`. Keep every pair.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_label_hard(s_a, s_b, margin):
    a_wins = ...
    keep = ...
    return a_wins, keep

def my_label_bt(s_a, s_b, generator=None):
    a_wins = ...                                # True with probability σ(s_a − s_b)
    return a_wins, torch.ones_like(a_wins)

In [ ]:
ex4_ok = s6.check_label_hard(globals().get("my_label_hard"))
ex4_ok = s6.check_label_bt(globals().get("my_label_bt")) and ex4_ok

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s6.check_label_hard(s6.ref_label_hard, strict=True)
s6.check_label_bt(s6.ref_label_bt, strict=True)
label_hard = mine_or_ref(globals(), "my_label_hard", ex4_ok, s6.ref_label_hard)
label_bt = mine_or_ref(globals(), "my_label_bt", ex4_ok, s6.ref_label_bt)

s_a, s_b = r_rm[[0, 1, 4, 3]], r_rm[[1, 3, 4, 2]]
a_wins, keep = label_hard(s_a, s_b, 0.5)
print("pairs (a, b)       (y0,y1) (y1,y3) (y4,y4) (y3,y2)")
print("RM gap s_a − s_b  ", [f"{x:+.1f}" for x in s_a - s_b])
print("hard: a wins      ", a_wins.tolist())
print("hard: keep (m=0.5)", keep.tolist())
print("BT: P(a wins)     ", [f"{x:.2f}" for x in torch.sigmoid(s_a - s_b)])

### Reference explanation

### Reference answer and mechanism

The hard rule is what most online-DPO and RLAIF pipelines do: the higher-scored answer is chosen,
and pairs whose gap is below a margin are discarded, because a small gap is mostly reward-model
noise. In this table the smallest gap between different answers is 0.5, so a margin of 0.5 drops
only pairs of two identical answers, like `(y4, y4)`. Those become common once an online policy
concentrates: they carry no preference at all. Use the absolute gap; `(s_a − s_b) >= margin` alone would silently drop every pair that `b`
wins.

The Bradley–Terry rule treats the reward model as a model of a noisy annotator: a gap of +1.0 gives
a 73% chance that `a` is chosen, not 100%. That is the labelling under which DPO's optimum is
exactly `π*`. The next cells show what the difference does over many steps.

### Prediction

## Exercise 5 · the KL-regularized optimum

Write `π*(y) ∝ π_ref(y)·exp(r(y)/β)` for a table of answers. Work in log space and normalise with a
softmax over the last dimension, so the same line also works for a `[P, K]` table (one row per
prompt), which Part D needs.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_kl_optimum(p_ref, r, beta):
    return ...                                  # one line

In [ ]:
ex5_ok = s6.check_kl_optimum(globals().get("my_kl_optimum"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s6.check_kl_optimum(s6.ref_kl_optimum, strict=True)
kl_optimum = mine_or_ref(globals(), "my_kl_optimum", ex5_ok, s6.ref_kl_optimum)

def kl(p, q):
    return float((p * (p / q).log()).sum())

BETA = 1.0
pi_star = kl_optimum(pi_ref, r_rm, BETA)
print("π_ref", [f"{x:.2f}" for x in pi_ref])
print("π*   ", [f"{x:.2f}" for x in pi_star], f"(β = {BETA})")
print(f"E[r̂]: π_ref {pi_ref @ r_rm:+.3f} → π* {pi_star @ r_rm:+.3f} | E[r*]: π_ref {pi_ref @ r_true:+.3f} → π* {pi_star @ r_true:+.3f} | "
      f"KL(π* ‖ π_ref) = {kl(pi_star, pi_ref):.3f}")

### Reference explanation

### Reference answer and mechanism

`log π_ref + r/β` is the unnormalised log-probability; `softmax` divides by `Z`. Multiplying by β
instead of dividing is the common slip: a small β must mean a *large* move away from `π_ref`.

At β = 1, `π*` raises `y0` (0.10 → 0.25) and also `y4` (0.05 → 0.34): `π*` is optimal **for the
reward model**, flaws included. Its true reward (+0.10) is better than `π_ref`'s (−0.48) but well
below what `y0` alone would give. Keep that in mind: an optimizer that reaches `π*` exactly has
done its job; whether `π*` is good depends on the feedback source.

### The DPO runner (given)

The offline dataset comes from another writer `μ` that never produces `y0` or `y4` (an older model
that never gives the detailed answer, and never the fluent wrong one). The online runner samples
both answers of each pair from the current policy. Both use the same reward model, the same labeller,
the same β, the same step size (plain SGD) and the same number of pairs, so the only difference is
where the pairs come from.

In [ ]:
mu = torch.tensor([0.0, 0.30, 0.20, 0.30, 0.0, 0.20])     # the offline writer: no y0, no y4

def dpo_tabular_loss(theta, chosen, rejected, keep, beta=BETA):
    h = torch.log_softmax(theta, -1) - pi_ref.log()       # log π/π_ref of each answer
    per_pair = -F.logsigmoid(beta * (h[chosen] - h[rejected]))
    return (per_pair * keep).sum() / keep.sum().clamp(min=1)

def run_dpo(source, labeller, n_pairs=4000, batch=16, lr=0.5, margin=0.5, seed=0):
    g = torch.Generator().manual_seed(seed)
    theta = pi_ref.log().clone().requires_grad_(True)
    opt = torch.optim.SGD([theta], lr=lr)
    offline = torch.multinomial(mu, 2 * n_pairs, replacement=True, generator=g).view(n_pairs, 2)
    log = []
    for start in range(0, n_pairs, batch):
        if source == "offline":
            y = offline[start:start + batch]                                         # fixed dataset
        else:
            pi = torch.softmax(theta, -1).detach()
            y = torch.multinomial(pi, 2 * batch, replacement=True, generator=g).view(batch, 2)  # current policy
        s_a, s_b = r_rm[y[:, 0]], r_rm[y[:, 1]]                                        # the RM scores both
        a_wins, keep = label_hard(s_a, s_b, margin) if labeller == "hard" else label_bt(s_a, s_b, g)
        chosen, rejected = torch.where(a_wins, y[:, 0], y[:, 1]), torch.where(a_wins, y[:, 1], y[:, 0])
        loss = dpo_tabular_loss(theta, chosen, rejected, keep.to(theta.dtype))
        opt.zero_grad(); loss.backward(); opt.step()
        p = torch.softmax(theta, -1).detach()
        log.append({"pairs": start + batch, "kl_to_opt": kl(p, pi_star), "true": float(p @ r_true),
                    "kl_ref": kl(p, pi_ref)})
    return pd.DataFrame(log), theta.detach()

### Prediction

## Predict 2 · offline or online: which reaches π*?

Run four versions for 4,000 pairs each: {offline, online} × {Bradley–Terry labels, hard labels with
margin 0.5}. (a) With Bradley–Terry labels, which ends closer to `π*`? Can offline DPO change
`θ[y0]` or `θ[y4]` at all? (b) With hard labels, does online DPO settle at `π*`? (c) Which version
ends with the highest **true** reward, and why is that not a contradiction?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
configs = [("offline", "bt"), ("online", "bt"), ("offline", "hard"), ("online", "hard")]
runs, thetas = {}, {}
for source, labeller in configs:
    logs = [run_dpo(source, labeller, seed=s) for s in range(4)]
    runs[source, labeller] = pd.concat([lg for lg, _ in logs]).groupby("pairs").mean()
    thetas[source, labeller] = [th for _, th in logs]
summary = pd.DataFrame({f"{s}, {l}": runs[s, l].iloc[-1] for s, l in configs}).T
summary["min KL to π* along the run"] = [runs[c]["kl_to_opt"].min() for c in configs]
print(summary.round(3).to_string())
# exact: offline pairs never contain y0 or y4, so their logits never receive a gradient
for th in thetas["offline", "bt"] + thetas["offline", "hard"]:
    assert torch.equal(th[[0, 4]], pi_ref.log()[[0, 4]])
print("offline runs: θ[y0] and θ[y4] are bit-identical to their starting values in all 8 runs")
assert runs["online", "bt"]["kl_to_opt"].iloc[-1] < 0.05 < runs["offline", "bt"]["kl_to_opt"].iloc[-1]
assert runs["online", "hard"]["kl_to_opt"].iloc[-1] > 3 * runs["online", "hard"]["kl_to_opt"].min()

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
styles = {("offline", "bt"): (GRAY, "-"), ("online", "bt"): (BLUE, "-"),
          ("offline", "hard"): (GRAY, "--"), ("online", "hard"): (ORANGE, "--")}
for c in configs:
    color, ls = styles[c]
    axes[0].plot(runs[c].index, runs[c]["kl_to_opt"], color=color, ls=ls, label=f"{c[0]}, {'BT' if c[1] == 'bt' else 'hard + margin'} labels")
    axes[1].plot(runs[c].index, runs[c]["true"], color=color, ls=ls)
axes[0].set_yscale("log"); axes[0].set_xlabel("pairs used"); axes[0].set_ylabel("KL(π ‖ π*) = (J* − J(π)) / β")
axes[0].set_title("distance to the KL-regularized optimum"); axes[0].legend(fontsize=8)
axes[1].axhline(float(pi_star @ r_true), color=BLUE, lw=0.8, ls=":", label="E_π*[r*]")
axes[1].axhline(float(pi_ref @ r_true), color=GRAY, lw=0.8, ls=":", label="E_π_ref[r*]")
axes[1].set_xlabel("pairs used"); axes[1].set_ylabel("E_π[r*]"); axes[1].set_title("true reward (mean of 4 seeds)")
axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

**(a) Online reaches `π*`; offline cannot.** With Bradley–Terry labels, online DPO ends within
about 0.01 nats of `π*`. Offline DPO stalls near 0.5: its pairs never contain `y0` or `y4`, and for a
softmax table the gradient of `log π(c) − log π(r)` with respect to the logit of any third answer `u`
is `−π(u) + π(u) = 0`. The run verifies it bit-for-bit: `θ[y0]` and `θ[y4]` never change. Offline
DPO can only reorder answers its dataset contains. This is the cost of being offline from D34: the
model never learns about answers it writes but the dataset doesn't contain.

**(b) Hard labels overshoot.** With deterministic labels, online DPO passes close to `π*` in the
first few hundred pairs and then moves *past* it, toward the reward model's top answer `y4`. A hard
label says "a beats b with probability 1", which corresponds to an infinite reward gap, so the loss
keeps pushing the margin up and β no longer fixes the distance from `π_ref`. (Azar et al., 2023,
make this point about DPO with deterministic preferences.) The margin filter only stops updates
between near-ties. In practice this is controlled by stopping early, by a few rounds instead of
convergence, or by losses that target a finite margin.

**(c) Highest true reward: offline with hard labels.** It collapses onto `y1` ("No, it's toxic"),
the best answer *in its dataset*, and the dataset happens to exclude the reward model's worst
mistake `y4`. That is luck about coverage, not a better method: the same run also can never reach
`y0`. Online DPO is the better **optimizer** of the stated objective; whether the objective is good
depends on the labeller. With this reward model, online DPO finds `y4` because it explores answers
the policy actually writes, which is exactly what makes it both stronger and more exposed to the
reward model's errors. Part C adds a second feedback source to block that error.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Offline vs online DPO: distance to the KL optimum and true reward, with Bradley–Terry and hard labels](../figures/chapter-13/study-path-s6_combinations-03.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part C · Mixing a reward model with a verifier

Parts A and B used one feedback source, and its errors went straight into the policy. A verifier
(S5) is the opposite kind of source: exact, but coarse. Here `v(y) = 1` for `y0` and `y1`, `0` for
the rest. It can't tell `y0` (detailed) from `y1` (terse), and it can't rank among wrong answers.
The reward model can rank everything, including, wrongly, `y4` first. Combining them should take
correctness from the verifier and preference among correct answers from the reward model.

Two problems come first.

1. **Scale.** A reward model's score has an arbitrary offset and scale (Bradley–Terry only constrains
   differences), while the verifier is 0/1. We **z-normalise** the scores within each GRPO group of
   `G` samples for the same prompt: `z = (r̂ − mean) / (std + ε)`.
2. **Combination rule.** A **weighted sum** `R = v + w·z` lets a large RM score outweigh the
   verifier. A **gate** `R = v·(1 + w·σ(z))` counts the reward model only when the verifier passes:
   correct answers get a reward in `(1, 1 + w)`, wrong answers get exactly 0.

DeepSeek-R1's last RL stage uses a third combination, **routing**: rule-based rewards for reasoning
prompts, a reward model for general prompts. Each prompt then has a single source, and nothing is mixed.

### Prediction

## Exercise 6 · z-normalise reward-model scores within each group

`s` has shape `[B, G]`: `B` groups (prompts), `G` sampled answers each. Return the z-score of each
row: subtract the row mean and divide by the row standard deviation plus `eps`, so a group whose
answers all got the same score returns zeros instead of NaN.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_group_z(s, eps=1e-6):
    return ...                                  # one line, per row (dim -1, keepdim=True)

In [ ]:
ex6_ok = s6.check_group_z(globals().get("my_group_z"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s6.check_group_z(s6.ref_group_z, strict=True)
group_z = mine_or_ref(globals(), "my_group_z", ex6_ok, s6.ref_group_z)

example = torch.tensor([[0, 1, 4, 4], [0, 4, 0, 4]])     # two groups of 4 sampled answers
print("answers    ", example.tolist())
print("RM scores  ", r_rm[example].tolist())
print("z in group ", [[round(x, 2) for x in row] for row in group_z(r_rm[example]).tolist()])

### Reference explanation

### Reference answer and mechanism

Normalising per row (`dim=-1`) matters: a global mean would mix groups from different prompts,
whose score levels can differ for reasons unrelated to the answers.

Look at the second group: `y0` and `y4` only. Whatever the actual RM gap (1.0 here), z-normalisation
maps the two answers to about −0.87 and +0.87. **Inside a group, z keeps the order and discards the
size of the gap.** A gap of 0.01 would give the same ±0.87. Keep this in mind for the next prediction.

### Prediction

## Exercise 7 · sum or gate

Write both combinations for tensors `v` (verifier, 0/1) and `z` (normalised RM scores) of the same
shape, and a weight `w ≥ 0`:

- `my_combine_sum(v, z, w)` returns `v + w·z`.
- `my_combine_gate(v, z, w)` returns `v·(1 + w·σ(z))`: zero when `v = 0`, between 1 and `1 + w` when `v = 1`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_combine_sum(v, z, w):
    return ...

def my_combine_gate(v, z, w):
    return ...

In [ ]:
ex7_ok = s6.check_combine(globals().get("my_combine_sum"), globals().get("my_combine_gate"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s6.check_combine(s6.ref_combine_sum, s6.ref_combine_gate, strict=True)
combine_sum = mine_or_ref(globals(), "my_combine_sum", ex7_ok, s6.ref_combine_sum)
combine_gate = mine_or_ref(globals(), "my_combine_gate", ex7_ok, s6.ref_combine_gate)

z_ex = group_z(r_rm[example].double())
v_ex = verifier[example]
print("sum, w=1 ", [[round(x, 2) for x in row] for row in combine_sum(v_ex, z_ex, 1.0).tolist()])
print("gate, w=1", [[round(x, 2) for x in row] for row in combine_gate(v_ex, z_ex, 1.0).tolist()])

### Reference explanation

### Reference answer and mechanism

The gate multiplies by `v`, so the reward-model term can't reach a wrong answer. Since `σ(z) ∈ (0, 1)`,
a correct answer's reward lies in `(1, 1 + w)` and every correct answer beats every wrong one,
whatever the reward model says (the check tests 2,000 random cases with `w = 3`). Among correct
answers, the reward model still decides the order.

The sum has no such guarantee. In the second example group (`y0, y4, y0, y4`), the sum gives `y0`
`1 − 0.87 = 0.13` and `y4` `0 + 0.87 = 0.87`: the wrong answer wins the group.

### A tabular GRPO trainer (given)

The same update rule for every reward: sample `B = 8` groups of `G = 16` answers from the current
table, compute a reward per sample, use the group-mean baseline `A = R − mean(R)` as the
**advantage** (how much better a sample did than its group), and step on `−mean(A·log π(y))`. Only the
reward function changes between runs. (This is S4's GRPO with the std division and clipping left out;
one update per batch makes the ratio exactly 1.)

In [ ]:
def pg_train(reward_fn, iters=300, B=8, G=16, lr=0.1, seed=0):
    g = torch.Generator().manual_seed(seed)
    theta = pi_ref.log().clone().requires_grad_(True)
    opt = torch.optim.Adam([theta], lr=lr)
    for _ in range(iters):
        logp = torch.log_softmax(theta, -1)
        y = torch.multinomial(logp.exp().detach(), B * G, replacement=True, generator=g).view(B, G)
        R = reward_fn(y)                                    # [B, G], no gradient
        A = R - R.mean(1, keepdim=True)                     # group-mean baseline
        loss = -(A * logp[y]).mean()
        opt.zero_grad(); loss.backward(); opt.step()
    return torch.softmax(theta, -1).detach()

W = 1.0
rewards = {
    "verifier only": lambda y: verifier[y],
    "RM only (z)": lambda y: group_z(r_rm[y]),
    f"sum, w = {W:g}": lambda y: combine_sum(verifier[y], group_z(r_rm[y]), W),
    f"gate, w = {W:g}": lambda y: combine_gate(verifier[y], group_z(r_rm[y]), W),
    "sum, w = 0.5": lambda y: combine_sum(verifier[y], group_z(r_rm[y]), 0.5),
}

### Prediction

## Predict 3 · which answer wins under each reward?

Train the table with each reward (3 seeds each, 300 updates). Where does the probability mass end
up under (a) the verifier alone, (b) the reward model alone, (c) the sum with `w = 1`, (d) the gate
with `w = 1`? For (c), use what Exercise 6 showed about a group that contains only `y0` and `y4`.
A fifth run, the sum with `w = 0.5`, is included for comparison.

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
final = {name: torch.stack([pg_train(fn, seed=s) for s in range(3)]) for name, fn in rewards.items()}
table = pd.DataFrame({name: ps.mean(0).numpy() for name, ps in final.items()}, index=[f"y{i}" for i in range(K)]).T
table["P(correct)"] = [float(ps.mean(0) @ verifier) for ps in final.values()]
table["E[r*]"] = [float(ps.mean(0) @ r_true) for ps in final.values()]
print(table.round(2).to_string())
assert table.loc[f"gate, w = {W:g}", "y0"] > 0.95 and table.loc[f"sum, w = {W:g}", "y4"] > 0.95

fig, ax = plt.subplots(figsize=(10, 3.2))
width = 0.16
for k, ((name, ps), color) in enumerate(zip(final.items(), [GREEN, ORANGE, RED, BLUE, GRAY])):
    ax.bar(np.arange(K) + (k - 2) * width, ps.mean(0), width, color=color, label=name)
ax.set_xticks(range(K), [f"y{i}\n{'correct' if verifier[i] else 'wrong'}" + ("\nRM's top" if i == 4 else "") for i in range(K)])
ax.set_ylabel("final probability (mean of 3 seeds)"); ax.set_title("tabular GRPO after 300 updates, by reward")
ax.set_ylim(0, 1.25); ax.legend(fontsize=8, ncols=5, loc="upper center")
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

(a) **Verifier only**: all mass on correct answers, split between `y0` and `y1` (in this run about
0.3 / 0.7). Both get reward 1, so within a group they have the same advantage and nothing prefers
one over the other; the split follows the starting probabilities and sampling noise.

(b) **RM only**: all mass on `y4`, the reward model's top answer, as in Parts A and B.

(c) **Sum, w = 1**: also `y4`, in all seeds. Early on, groups contain many answers and `y0` wins. But as
the policy concentrates on `y0` and `y4`, a group contains mostly those two, and z-normalisation maps
them to about ±1 regardless of the real gap. Then `R(y0) = 1 − w·1 = 0` and `R(y4) = 0 + w·1 = 1`.
The RM term's effective weight grows exactly when it matters. With `w = 0.5` the two tie in such a
group (`1 − 0.5 = 0.5` each), and the result depends on the seed: the fifth row ends split between
`y0` and `y4` on average over the three seeds.

(d) **Gate, w = 1**: all mass on `y0`. The verifier removes `y2–y5`; among correct answers, the
reward model prefers `y0` to `y1`, and that preference now decides. This is the combination we
wanted: correctness from the exact source, ranking from the graded one.

The general rule: **a learned score added to a verifier needs a bound** on how much it can
contribute. Gating gives that bound by construction; a sum gives it only if `w` times the largest
possible z-difference stays below the verifier's step, and group z-scores make that hard to guarantee.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Final policies under verifier-only, RM-only, summed and gated rewards](../figures/chapter-13/study-path-s6_combinations-04.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

---

# Part D · A three-stage pipeline: SFT → DPO → RLVR

Tülu 3 (Lambert et al., 2024) trains in three stages, each with a different feedback source:

| Stage | Training data | Feedback source | Update rule | Reference (KL anchor) |
|---|---|---|---|---|
| 1 · SFT | demonstrations | none: imitate | NLL (Exercise 3) | — |
| 2 · DPO | pairs, partly sampled from the SFT model, labelled by an LLM judge | preference labels | DPO loss | the SFT model |
| 3 · RLVR | prompts with checkable answers | verifier | policy gradient (Tülu 3 uses PPO) | the DPO model |

The last column is the mechanism this part is about. **Each stage is anchored to the output of the
previous one**, so a later stage pays a KL cost for undoing earlier work.

### The two-prompt toy

A math prompt, *What is 17 × 24?* (408), and a chat prompt, *How do I ask my manager for a day
off?*, six answers each. The **base** model (pretrained, not chat-tuned) mostly repeats or continues
the prompt. **Demonstrations** come from annotators: well-formatted, sometimes with an arithmetic
slip. **Preference labels** come from people who like shown work and politeness but don't check the
arithmetic: they score `m4` (wrong, with steps) as highly as `m3` (right, with steps). Only the
math prompt has a **verifier**.

In [ ]:
math_answers = ["17 × 24 = 408.", "408", "17 × 24 = 418.",
                "17 × 20 + 17 × 4 = 340 + 68 = 408.", "17 × 20 + 17 × 4 = 340 + 78 = 418.",
                "What is 17 × 24? What is 17 × 25? What is"]
chat_answers = ["Send a short, polite message with the date and who covers your work.", "Just tell them.",
                "Here is a template: 'Hi Sam, could I take Friday 14th off? Alex will cover ...'",
                "As an AI, I can't help with workplace requests.", "Don't ask. Just don't show up.",
                "How do I ask my manager for a raise? How do I ask"]
base  = torch.tensor([[0.08, 0.10, 0.12, 0.05, 0.15, 0.50],      # row 0: math prompt
                      [0.10, 0.15, 0.05, 0.10, 0.10, 0.50]])     # row 1: chat prompt
demo  = torch.tensor([[0.45, 0.10, 0.05, 0.30, 0.10, 0.00],      # annotators: formatted, 15% slips
                      [0.60, 0.10, 0.30, 0.00, 0.00, 0.00]])
r_pref = torch.tensor([[1.0, 0.0, 1.0, 2.0, 2.0, -2.0],          # people like shown work, don't check it
                       [2.0, 0.0, 2.5, -1.0, -2.0, -3.0]])
v_math = torch.tensor([1.0, 1.0, 0.0, 1.0, 0.0, 0.0])            # the verifier: is 408 the result?
print(pd.DataFrame({"math answer": math_answers, "base": base[0], "demo": demo[0], "r_pref": r_pref[0],
                    "verifier": v_math.int()}).to_string())
print()
print(pd.DataFrame({"chat answer": chat_answers, "base": base[1], "demo": demo[1], "r_pref": r_pref[1]}).to_string())

### Prediction

## Exercise 8 · the RLVR loss, anchored to the previous stage

Write the loss of the RLVR stage for one prompt with a tabular policy:

`L = −mean_{b,g}[A_{b,g}·log π(y_{b,g})] + β·KL(π ‖ π_ref)`,  with `A = R − mean_g(R)` per group.

`logits` has shape `[K]`, `ref_logp` `[K]` is the log-probability table of the anchor (the DPO
model), `y` and `R` have shape `[B, G]`. For a table, the KL is exact:
`KL(π ‖ π_ref) = Σ_y π(y)·(log π(y) − log π_ref(y))`.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_rlvr_loss(logits, ref_logp, y, R, beta):
    logp = torch.log_softmax(logits, -1)
    A = ...                                     # group-mean baseline, shape [B, G]
    kl = ...                                    # exact KL(π ‖ π_ref), a scalar
    return ...                                  # policy-gradient term + beta * kl

In [ ]:
ex8_ok = s6.check_rlvr_loss(globals().get("my_rlvr_loss"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s6.check_rlvr_loss(s6.ref_rlvr_loss, strict=True)
rlvr_loss = mine_or_ref(globals(), "my_rlvr_loss", ex8_ok, s6.ref_rlvr_loss)

g = torch.Generator().manual_seed(1)
# Stage 1 · SFT on 500 demonstrations per prompt (sft_loss from Exercise 3), starting from the base model
demos = torch.stack([torch.multinomial(demo[i], 500, replacement=True, generator=g) for i in range(2)])
theta = base.log().clone().requires_grad_(True)
opt = torch.optim.Adam([theta], lr=0.05)
for _ in range(150):
    loss = sft_loss(theta[0], demos[0]) + sft_loss(theta[1], demos[1])
    opt.zero_grad(); loss.backward(); opt.step()
p_sft = torch.softmax(theta, -1).detach()

# Stage 2 · DPO on pairs sampled from the SFT model, Bradley–Terry labels from r_pref, anchored to SFT
BETA_DPO = 0.5
ref2 = p_sft.log()
theta = ref2.clone().requires_grad_(True)
opt = torch.optim.SGD([theta], lr=0.5)
for _ in range(300):
    y = torch.stack([torch.multinomial(p_sft[i], 32, replacement=True, generator=g).view(16, 2) for i in range(2)])
    s = torch.stack([r_pref[i][y[i]] for i in range(2)])                           # [2, 16, 2]
    a_wins, _ = label_bt(s[..., 0], s[..., 1], g)
    c, r = torch.where(a_wins, y[..., 0], y[..., 1]), torch.where(a_wins, y[..., 1], y[..., 0])
    h = torch.log_softmax(theta, -1) - ref2
    loss = -F.logsigmoid(BETA_DPO * (h.gather(1, c) - h.gather(1, r))).mean()
    opt.zero_grad(); loss.backward(); opt.step()
p_dpo = torch.softmax(theta, -1).detach()

# Stage 3 · RLVR on the math prompt only (rlvr_loss), anchored to the DPO model
BETA_RL = 0.2
theta = p_dpo.log().clone().requires_grad_(True)
opt = torch.optim.Adam([theta], lr=0.05)
for _ in range(400):
    y = torch.multinomial(torch.softmax(theta[0], -1).detach(), 8 * 16, replacement=True, generator=g).view(8, 16)
    loss = rlvr_loss(theta[0], p_dpo[0].log(), y, v_math[y], BETA_RL)
    opt.zero_grad(); loss.backward(); opt.step()
p_rl = torch.softmax(theta, -1).detach()

stages = {"base": base, "SFT": p_sft, "DPO": p_dpo, "RLVR": p_rl}
def kl_rows(p, q):
    return (p * (p / q).log()).sum(-1)
rows, prev = [], None
for name, p in stages.items():
    rows.append({"stage": name, "P(correct) math": float(p[0] @ v_math), "E[r_pref] math": float(p[0] @ r_pref[0]),
                 "E[r_pref] chat": float(p[1] @ r_pref[1]), "P(m4: wrong, with steps)": float(p[0, 4]),
                 "KL to prev. (math)": float(kl_rows(p, prev)[0]) if prev is not None else float("nan"),
                 "KL to prev. (chat)": float(kl_rows(p, prev)[1]) if prev is not None else float("nan")})
    prev = p
pipeline = pd.DataFrame(rows).set_index("stage")
print(pipeline.round(3).to_string())

### Reference explanation

### Reference answer and mechanism

`A = R − mean(R)` per group is the group-mean baseline from S4: a sample's advantage is how much
better it did than the other samples for the same prompt. `Σ π·(log π − log π_ref)` is the exact KL
of a table (S2, D35). Two common slips: no baseline (the expected direction is about the same, but much
noisier) and the reversed KL `Σ π_ref·(log π_ref − log π)`, which penalises a different thing.

Read the stage table:

- **SFT** moves the most (KL ≈ 1.1–1.4 nats per prompt): it replaces the base model's repetitions with
  the demonstrated formats. P(correct) on math goes from 0.23 to about 0.84, close to the annotators' own
  85% (they slip on 15% of demonstrations).
- **DPO** raises the preferred style on both prompts. On math it raises *shown work*, `m3` and the
  wrong `m4` alike (`m4`: 0.10 → 0.13). P(correct) stays at about 0.84: the preference labels never
  checked the arithmetic.
- **RLVR** raises P(correct) on math to about 1.00 and removes `m2` and `m4` (0.13 → about 0.002). On chat its KL is exactly 0: a
  table has separate parameters for each prompt, so training on math can't move chat. In a network
  with shared weights it can, in either direction, which is why pipelines re-evaluate every capability
  after each stage.

### What the anchor preserves

The RLVR stage maximises `E_π[v] − β·KL(π ‖ π_DPO)`. By Exercise 5 its optimum is
`π*_RL ∝ π_DPO·exp(v/β)`. Since `v` is the same (1) for every correct answer, the factor `exp(1/β)` is
the same for all of them, so **the ratios among correct answers are exactly those of the DPO model**.
The verifier decides *whether* an answer is right; the earlier stages still decide *which* right
answer is preferred. The next cell checks the trained table against this closed form and draws all
four stages.

In [ ]:
opt_rl = kl_optimum(p_dpo[0], v_math, BETA_RL)
correct = v_math.bool()
print("π_RLVR (trained)  ", [f"{x:.3f}" for x in p_rl[0]])
print("π*_RL (closed form)", [f"{x:.3f}" for x in opt_rl], f"| KL(trained ‖ closed form) = {kl(p_rl[0], opt_rl):.5f}")
print("ratios among correct answers (m0, m1, m3): DPO", [f"{x:.3f}" for x in p_dpo[0, correct] / p_dpo[0, correct].sum()],
      "| RLVR", [f"{x:.3f}" for x in p_rl[0, correct] / p_rl[0, correct].sum()])
assert kl(p_rl[0], opt_rl) < 0.01
torch.testing.assert_close(p_rl[1], p_dpo[1])                    # chat prompt: untouched by RLVR

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
colors = [GRAY, BLUE, ORANGE, GREEN]
for j, (ax, labels, title) in enumerate(zip(axes, [["m0 ✓", "m1 ✓", "m2 ✗", "m3 ✓", "m4 ✗", "m5 ✗"], [f"c{i}" for i in range(6)]],
                                            ["math prompt (✓ = verifier passes)", "chat prompt (no verifier)"])):
    for k, ((name, p), color) in enumerate(zip(stages.items(), colors)):
        ax.bar(np.arange(6) + (k - 1.5) * 0.2, p[j], 0.2, color=color, label=name)
    ax.set_xticks(range(6), labels); ax.set_title(title); ax.set_ylabel("probability")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![SFT → DPO → RLVR on two prompts: what each stage changes](../figures/chapter-13/study-path-s6_combinations-05.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

The trained RLVR table matches the closed form to within about 0.0005 nats, and the ratios among
`m0`, `m1`, `m3` are the DPO model's ratios to two decimals. Without the anchor (β → 0), any split among
correct answers would be optimal, and RLVR could drift to the terse `m1` and undo DPO's preference for
shown work. That is the job of the KL term in a multi-stage pipeline: **each stage is anchored to the
output of the one before**.

Why this order? SFT first, because preference and verifier signals need a model that already
produces answers in the expected format (the base model is correct only 23% of the time, mostly by
repeating the question). Verifiable RL last, because it covers only the prompts that have a checker,
and its anchor keeps what DPO added to the others.

---

# Part E · Qwen3: one round of reward-model-labelled online DPO

Now the combination from Part B on a real model: **a reward model labels pairs that the policy
itself wrote, and DPO trains on them**. One round has four steps.

1. **Feedback source.** Train a reward model `RM_A` on 2,000 HH-RLHF pairs (S1; the loop is given).
   Train a second one, `RM_B`, on 2,000 *different* pairs. `RM_A` labels; `RM_B` is used only to
   evaluate, as a second opinion that did not see `RM_A`'s training pairs.
2. **Sample.** Qwen3-0.6B, `π_0`, writes `n = 4` answers to each of 160 training questions. `π_0` is
   also the frozen DPO reference.
3. **Label.** `RM_A` scores every answer. The best and the worst answer of a prompt form a pair if
   their score gap is at least a margin (Exercise 9). This is the pair construction of Self-Rewarding
   LMs, with a separate reward model in place of the model judging itself.
4. **Optimize.** DPO on those pairs, anchored to `π_0`.

Then we evaluate on 96 held-out test questions: the scores that both reward models give to fresh
samples before and after, with paired standard errors (Exercise 10); answer length; and the DPO
implicit-reward accuracy on held-out `RM_A`-labelled pairs.

In [ ]:
tok = Q.load_tokenizer()
train_pairs, test_pairs = Q.hh_pairs("train"), Q.hh_pairs("test")
train_questions = Q.hh_questions("train", start=20000)       # never used by either reward model
test_questions = Q.hh_questions("test")
print(f"HH helpful-base pairs: {len(train_pairs):,} train, {len(test_pairs):,} test | "
      f"single-turn questions: {len(train_questions):,} from train[20000:], {len(test_questions):,} test")
print(inspect.getsource(Q.render))

### Two reward models (given: S1 material)

Each reward model is Qwen3-0.6B with its language-model head replaced by a 1-output linear layer
(`Q.load_reward_model`, D30), read at the last token (`<|im_end|>`, D28) and trained with the
Bradley–Terry loss `−log σ(r(chosen) − r(rejected))` for one pass over its 2,000 pairs. The top
`RM_TOP` = 28 of the 28 transformer layers, the final norm and the head are trained; the
embeddings stay frozen. Master weights are float32 and the forward pass runs in bfloat16 autocast.
Both models together take about 7 minutes.

In [ ]:
AUTOCAST = lambda: torch.autocast("cuda", dtype=torch.bfloat16)
N_RM, RM_BS, RM_LR, RM_TOP, RM_MAXLEN = 2000, 8, 1e-5, 28, 256

def train_reward_model(pairs, seed):
    """One pass of Bradley–Terry training over `pairs`; returns the frozen model and its loss curve."""
    torch.manual_seed(seed)
    rm = Q.load_reward_model(tok, dtype=torch.float32)
    rm.requires_grad_(False)                                   # train the top RM_TOP layers + norm + head
    for module in [*rm.model.layers[-RM_TOP:], rm.model.norm, rm.score]:
        module.requires_grad_(True)
    opt = torch.optim.AdamW([p for p in rm.parameters() if p.requires_grad], lr=RM_LR, weight_decay=0.0)
    # batches of similar length waste less compute on padding; the batch order is random
    order = sorted(range(len(pairs)), key=lambda i: len(str(pairs[i]["prompt"])) + len(pairs[i]["chosen"]) + len(pairs[i]["rejected"]))
    batches = [order[i:i + RM_BS] for i in range(0, len(order), RM_BS)]
    random.Random(seed).shuffle(batches)
    rm.train(); losses = []
    for idx in batches:
        batch = [pairs[i] for i in idx]
        texts = ([Q.render(tok, p["prompt"], p["chosen"]) for p in batch] +
                 [Q.render(tok, p["prompt"], p["rejected"]) for p in batch])
        enc = tok(texts, return_tensors="pt", padding=True, padding_side="right", truncation=True,
                  max_length=RM_MAXLEN).to(DEVICE)
        with AUTOCAST():
            scores = rm(**enc).logits.squeeze(-1).float()      # [2B]: chosen scores, then rejected
        loss = -F.logsigmoid(scores[:len(batch)] - scores[len(batch):]).mean()
        loss.backward(); torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0); opt.step(); opt.zero_grad()
        losses.append(loss.item())
    del opt; Q.free()
    return rm.to(torch.bfloat16).eval().requires_grad_(False), losses

@torch.no_grad()
def pair_accuracy(rm, pairs):
    """Fraction of pairs where the chosen answer scores higher, and its standard error."""
    s_c = Q.rm_scores(rm, tok, [p["prompt"] for p in pairs], [p["chosen"] for p in pairs], max_length=RM_MAXLEN)
    s_r = Q.rm_scores(rm, tok, [p["prompt"] for p in pairs], [p["rejected"] for p in pairs], max_length=RM_MAXLEN)
    wins = (s_c > s_r).double()
    return wins.mean().item(), (wins.std() / math.sqrt(len(wins))).item()

random.seed(0)
rm_pairs_a = random.sample(train_pairs[:10000], N_RM)          # RM_A: the labeller
rm_pairs_b = random.sample(train_pairs[10000:20000], N_RM)     # RM_B: disjoint pairs, evaluation only
rm_eval_pairs = test_pairs[:400]
torch.cuda.reset_peak_memory_stats(); t0 = time.time()
rm_a, loss_a = train_reward_model(rm_pairs_a, seed=0)
rm_b, loss_b = train_reward_model(rm_pairs_b, seed=1)
acc_a, acc_b = pair_accuracy(rm_a, rm_eval_pairs), pair_accuracy(rm_b, rm_eval_pairs)
rm_peak = Q.gib(torch.cuda.max_memory_allocated())
print(f"trained two reward models in {time.time() - t0:.0f}s | peak GPU memory {rm_peak:.1f} GiB")
print(f"held-out accuracy on {len(rm_eval_pairs)} test pairs (± 1 SE): RM_A {acc_a[0]:.1%} ± {acc_a[1]:.1%} | "
      f"RM_B {acc_b[0]:.1%} ± {acc_b[1]:.1%} | chance 50%")

These are **weak** reward models, as expected from 2,000 noisy pairs and one pass: in the reference
runs both agree with the human label on 54–56% of held-out pairs (± 2.5% SE), about two standard
errors above chance. Reward models far larger than ours reach only about 70% on HH, partly because the
human labels are themselves noisy. So `RM_A` disagrees with the human label on about 45% of pairs.
Keep that number in mind for every result below: online DPO will faithfully optimize whatever `RM_A`
prefers, and with an accuracy this low, much of what it prefers is not what the annotators preferred.

### Sampling from the policy (given)

`Q.rollout` samples one answer per question from the policy itself at temperature 1 with no
top-k or top-p, so the samples are on-policy (glossary: **on-policy** = generated by the current
policy). `Q.score_rollout` scores the sampled token ids directly with a reward model; the policy
and the reward models share the tokenizer. Each question gets a short length instruction so that
most answers finish within the token budget; truncated answers are excluded from pairs below.

In [ ]:
print(inspect.getsource(Q.rollout))
print(inspect.getsource(Q.score_rollout))

SUFFIX, MAX_NEW, N_TRAIN_Q, N_SAMPLES, N_TEST_Q = " Answer in two or three sentences.", 96, 160, 4, 96
as_messages = lambda q: [{"role": "user", "content": q + SUFFIX}]
dpo_questions = [as_messages(q) for q in train_questions[:N_TRAIN_Q]]
eval_questions = [as_messages(q) for q in test_questions[:N_TEST_Q]]

policy = Q.load_chat_model(dtype=torch.float32)                         # π_θ, trained
ref = Q.load_chat_model(dtype=torch.float32).eval().requires_grad_(False)   # π_0, frozen reference

@torch.no_grad()
def sample_scored(model, questions, n, seed, bs=128):
    """n samples per question, scored by both reward models; every tensor has shape [len(questions), n]."""
    torch.manual_seed(seed); model.eval()
    flat = [q for q in questions for _ in range(n)]
    texts, s_a, s_b, length, trunc = [], [], [], [], []
    for i in range(0, len(flat), bs):
        with AUTOCAST():
            ro = Q.rollout(model, tok, flat[i:i + bs], max_new=MAX_NEW)
            s_a.append(Q.score_rollout(rm_a, ro)); s_b.append(Q.score_rollout(rm_b, ro))
        texts += Q.response_texts(tok, ro); length.append(ro["mask"].sum(1)); trunc.append(ro["truncated"])
    shape = lambda x: torch.cat(x).float().cpu().view(len(questions), n)
    return {"texts": [texts[i * n:(i + 1) * n] for i in range(len(questions))], "rm_a": shape(s_a),
            "rm_b": shape(s_b), "length": shape(length), "truncated": shape(trunc).bool()}

t0 = time.time()
train_samples = sample_scored(policy, dpo_questions, N_SAMPLES, seed=1)
print(f"{N_TRAIN_Q} questions × {N_SAMPLES} samples in {time.time() - t0:.0f}s | mean length "
      f"{train_samples['length'].mean():.0f} tokens | truncated {train_samples['truncated'].float().mean():.0%}")
i = 0
print("\nQ:", dpo_questions[i][0]["content"])
for text, score in zip(train_samples["texts"][i], train_samples["rm_a"][i]):
    print(f"  RM_A {score:+.2f} | {text[:150]!r}")

### Prediction

## Exercise 9 · best-vs-worst pairs with a margin

`scores` has shape `[P, n]`: `RM_A`'s score for each of the `n` samples of each prompt. Return three
tensors of shape `[P]`: the index of the highest-scored sample (`best`), the index of the
lowest-scored sample (`worst`), and `keep`, true when `best`'s score minus `worst`'s score is at
least `margin`. The best and worst samples are the most informative pair the four samples can
form, and the margin drops prompts where the reward model sees no clear difference.

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_best_vs_worst(scores, margin):
    best = ...
    worst = ...
    keep = ...
    return best, worst, keep

In [ ]:
ex9_ok = s6.check_best_vs_worst(globals().get("my_best_vs_worst"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s6.check_best_vs_worst(s6.ref_best_vs_worst, strict=True)
best_vs_worst = mine_or_ref(globals(), "my_best_vs_worst", ex9_ok, s6.ref_best_vs_worst)

def build_pairs(questions, texts, scores, truncated, margin):
    """Online DPO pairs: best vs worst answer per prompt by RM_A; skip prompts where either is truncated."""
    best, worst, keep = best_vs_worst(scores, margin)
    rows = torch.arange(len(questions))
    keep = keep & ~truncated[rows, best] & ~truncated[rows, worst]
    return [{"prompt": q, "chosen": texts[i][best[i]], "rejected": texts[i][worst[i]],
             "gap": float(scores[i, best[i]] - scores[i, worst[i]])} for i, q in enumerate(questions) if keep[i]]

MARGIN = round(0.5 * train_samples["rm_a"].std().item(), 2)        # half a standard deviation of RM_A's scores
dpo_pairs = build_pairs(dpo_questions, train_samples["texts"], train_samples["rm_a"], train_samples["truncated"], MARGIN)
gaps = torch.tensor([p["gap"] for p in dpo_pairs])
print(f"kept {len(dpo_pairs)} of {N_TRAIN_Q} prompts (margin {MARGIN}) | best − worst RM_A gap of kept pairs: median "
      f"{gaps.median():.2f}, min {gaps.min():.2f} | std of all RM_A scores {train_samples['rm_a'].std():.2f}")
p = dpo_pairs[0]
print("\nQ:", p["prompt"][0]["content"])
print("chosen  :", repr(p["chosen"][:200]))
print("rejected:", repr(p["rejected"][:200]))

### Reference explanation

### Reference answer and mechanism

`argmax(1)` and `argmin(1)` pick the extreme samples of each prompt, and `max − min` is the largest
gap among the four. Using the gap between the best and the *second* best would keep a different set
of prompts and is the slip the check catches.

`build_pairs` then drops prompts whose best or worst answer was truncated at the token budget:
DPO appends `<|im_end|>` to every answer, so a truncated answer would teach the model to stop
mid-sentence (or, as the rejected answer, teach it that stopping is bad).

Read the example pair yourself. With a reward model this weak, many chosen answers won't look
better to you. That is the honest state of the labels this round trains on.

### DPO on the labelled pairs (given: S2 material)

`Q.answer_logps` returns `Σ log π(answer tokens + <|im_end|>)` for the chosen and rejected answer of
each pair, shape `[B, 2]`. The DPO loss is Bradley–Terry on the implicit reward
`r̂(y) = β·(log π_θ(y) − log π_0(y))`. Two passes over the pairs, batch 8, β = 0.1, AdamW at a small
learning rate. The trained policy keeps float32 master weights (the forward pass runs in bfloat16
autocast): in bfloat16 a weight near 0.02 has a spacing of about 1.2·10⁻⁴, so AdamW steps of
10⁻⁶–10⁻⁵ would round back to the old value and most updates would be lost. The **implicit-reward accuracy** on held-out pairs (test questions, `π_0`'s own
samples, labelled by `RM_A` with the same rule) measures whether the update generalizes beyond the
training pairs; before training every implicit reward is exactly 0 and ties count as ½.

In [ ]:
print(inspect.getsource(Q.answer_logps))
DPO_BETA, DPO_LR, DPO_EPOCHS, DPO_BS = 0.1, 3e-6, 2, 8

def dpo_loss(policy_lp, ref_lp, beta=DPO_BETA):
    implicit = beta * (policy_lp - ref_lp)                     # [B, 2]: implicit reward of chosen, rejected
    return -F.logsigmoid(implicit[:, 0] - implicit[:, 1]).mean(), implicit

@torch.no_grad()
def implicit_accuracy(pairs):
    policy.eval(); out = []
    for i in range(0, len(pairs), DPO_BS):
        with AUTOCAST():
            _, imp = dpo_loss(Q.answer_logps(policy, tok, pairs[i:i + DPO_BS]), Q.answer_logps(ref, tok, pairs[i:i + DPO_BS]))
        out.append(imp)
    imp = torch.cat(out)
    wins = (imp[:, 0] > imp[:, 1]).double() + 0.5 * (imp[:, 0] == imp[:, 1]).double()
    return wins.mean().item(), (wins.std() / math.sqrt(len(wins))).item()

t0 = time.time()
before = sample_scored(policy, eval_questions, 8, seed=2)       # π_0 on held-out questions, 8 samples each
heldout_pairs = build_pairs(eval_questions, [t[:4] for t in before["texts"]], before["rm_a"][:, :4],
                            before["truncated"][:, :4], MARGIN)          # same rule, first 4 samples
acc0 = implicit_accuracy(heldout_pairs)
print(f"held-out: {N_TEST_Q} questions × 8 samples from π_0 in {time.time() - t0:.0f}s | {len(heldout_pairs)} RM_A-labelled pairs "
      f"| implicit-reward accuracy before training {acc0[0]:.1%} (all ties)")

opt = torch.optim.AdamW(policy.parameters(), lr=DPO_LR, weight_decay=0.0)
torch.cuda.reset_peak_memory_stats(); dpo_log, t0 = [], time.time()
rng = random.Random(0)
for epoch in range(DPO_EPOCHS):
    policy.train()
    order = rng.sample(dpo_pairs, len(dpo_pairs))
    for start in range(0, len(order), DPO_BS):
        batch = order[start:start + DPO_BS]
        with torch.no_grad(), AUTOCAST():
            ref_lp = Q.answer_logps(ref, tok, batch)
        with AUTOCAST():
            loss, implicit = dpo_loss(Q.answer_logps(policy, tok, batch), ref_lp)
        loss.backward(); torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0); opt.step(); opt.zero_grad()
        dpo_log.append({"epoch": epoch, "loss": loss.item(), "chosen": implicit[:, 0].mean().item(),
                        "rejected": implicit[:, 1].mean().item()})
    last = pd.DataFrame(dpo_log).query("epoch == @epoch")
    print(f"epoch {epoch}: mean loss {last['loss'].mean():.3f} | implicit reward chosen {last['chosen'].mean():+.3f}, "
          f"rejected {last['rejected'].mean():+.3f} | {time.time() - t0:.0f}s")
dpo_peak = Q.gib(torch.cuda.max_memory_allocated())
del opt; Q.free()
acc1 = implicit_accuracy(heldout_pairs)
with torch.no_grad():
    n_changed = sum((p != q).sum().item() for p, q in zip(policy.parameters(), ref.parameters()))
print(f"fraction of policy weights changed by DPO: {n_changed / sum(p.numel() for p in policy.parameters()):.1%} (float32 master weights)")
print(f"implicit-reward accuracy on {len(heldout_pairs)} held-out pairs: {acc0[0]:.1%} → {acc1[0]:.1%} ± {acc1[1]:.1%} "
      f"| peak GPU memory during DPO {dpo_peak:.1f} GiB")

### Prediction

## Exercise 10 · a paired standard error

To compare the policy before and after, we score fresh samples for the **same** held-out questions
from both. Prompts differ a lot in how high any answer scores, so compare per prompt: with
`before` and `after` of shape `[P]` (one mean score per prompt), return the mean of
`d = after − before` and its standard error `std(d) / √P`, using the sample standard deviation
(`d.std()`, which divides by `P − 1`).

Write your lines in the next cell (replace each `...`), run it, then run the check.

### Checkpoint

The check prints ✅ or ❌ with a hint. Retry until it passes, then compare with the reference.

In [ ]:
def my_paired_diff(before, after):
    d = ...
    return d.mean(), ...                        # mean change and its standard error

In [ ]:
ex10_ok = s6.check_paired_diff(globals().get("my_paired_diff"))

### Reference solution

Open this after your own attempt passes, or when you're stuck.

In [ ]:
s6.check_paired_diff(s6.ref_paired_diff, strict=True)
paired_diff = mine_or_ref(globals(), "my_paired_diff", ex10_ok, s6.ref_paired_diff)

# a null comparison: π_0's samples 1–4 against its own samples 5–8 on the same questions (true change = 0)
m, se = paired_diff(before["rm_a"][:, :4].mean(1), before["rm_a"][:, 4:].mean(1))
unpaired = math.sqrt(before["rm_a"][:, :4].mean(1).var() / N_TEST_Q + before["rm_a"][:, 4:].mean(1).var() / N_TEST_Q)
print(f"π_0 vs π_0 (RM_A): {m:+.3f} ± {se:.3f} (paired SE) | the unpaired SE would be {unpaired:.3f}")

### Reference explanation

### Reference answer and mechanism

Pairing removes the variation *between* prompts: some questions get high scores whatever the
answer, others low. That variation appears in both `before` and `after` and cancels in `d`. In the
reference run the paired SE is somewhat smaller than the unpaired one; the gain grows with how much
prompts differ.

The null comparison is a calibration check on the procedure itself: two sets of samples from the
same policy should differ by about 0 ± 1 SE, and here they do. Any before/after difference must be
read against this noise.

### Prediction

## Predict 4 · what did one round change?

Sample 4 fresh answers per held-out question from the trained policy `π_1`. Compared with `π_0` on
the same questions: (a) does `RM_A`'s mean score rise by more than 2 standard errors? (b) Does
`RM_B`, which never saw `RM_A`'s pairs, agree? (c) Does the answer length change? (d) If both reward
models rise, is that evidence that the answers became more helpful?

Pause and write your answer before running the next cell.

### Checkpoint

Record your reasoning before the reference; compare it with the computation and explanation.

### Reference solution

Run this reference after your prediction.

In [ ]:
t0 = time.time()
after = sample_scored(policy, eval_questions, 4, seed=3)
print(f"{N_TEST_Q} questions × 4 samples from π_1 in {time.time() - t0:.0f}s\n")
rows = []
for key, label in [("rm_a", "RM_A score (the labeller)"), ("rm_b", "RM_B score (held-out RM)"),
                   ("length", "answer length (tokens)"), ("truncated", "truncated fraction")]:
    b, a = before[key][:, :4].float().mean(1), after[key].float().mean(1)
    m, se = paired_diff(b, a)
    rows.append({"metric": label, "π_0": b.mean().item(), "π_1": a.mean().item(), "change": float(m),
                 "SE": float(se), "change / SE": float(m / se) if se > 0 else float("nan")})
online_dpo = pd.DataFrame(rows).set_index("metric")
print(online_dpo.round(3).to_string())
print(f"\nimplicit-reward accuracy on held-out pairs: {acc1[0]:.1%} ± {acc1[1]:.1%}")

# best-of-n on π_0's 8 held-out samples: select with RM_A, read both reward models
bon_rows = []
for n in [1, 2, 4, 8]:
    pick = before["rm_a"][:, :n].argmax(1, keepdim=True)
    sel_a, sel_b = before["rm_a"][:, :n].gather(1, pick).squeeze(1), before["rm_b"][:, :n].gather(1, pick).squeeze(1)
    bon_rows.append({"n": n, "RM_A": sel_a.mean().item(), "RM_A SE": (sel_a.std() / math.sqrt(N_TEST_Q)).item(),
                     "RM_B": sel_b.mean().item(), "RM_B SE": (sel_b.std() / math.sqrt(N_TEST_Q)).item()})
bon_real = pd.DataFrame(bon_rows).set_index("n")
print("\nbest-of-n with RM_A on π_0's samples (scores of the selected answer):")
print(bon_real.round(3).to_string())
print(f"\npeak GPU memory: reward models {rm_peak:.1f} GiB, DPO {dpo_peak:.1f} GiB")
for i in range(2):
    print("\nQ:", eval_questions[i][0]["content"])
    print("  π_0:", repr(before["texts"][i][0][:160]))
    print("  π_1:", repr(after["texts"][i][0][:160]))

corr = lambda x, y: float(np.corrcoef(x.flatten(), y.flatten())[0, 1])
print(f"\ncorrelation with answer length over π_0's {before['length'].numel()} held-out samples: "
      f"RM_A {corr(before['rm_a'], before['length']):+.2f} | RM_B {corr(before['rm_b'], before['length']):+.2f} | "
      f"RM_A vs RM_B {corr(before['rm_a'], before['rm_b']):+.2f}")

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
log = pd.DataFrame(dpo_log)
ax = axes[0]
ax.plot(log["chosen"], color=GREEN, label="chosen")
ax.plot(log["rejected"], color=RED, label="rejected")
ax.axhline(0, color=GRAY, lw=0.8)
ax.set_xlabel(f"DPO step ({DPO_EPOCHS} passes over {len(dpo_pairs)} pairs)"); ax.set_ylabel("implicit reward β·log π_θ/π_0")
ax.set_title("online DPO on RM_A-labelled pairs"); ax.legend(fontsize=8)
ax = axes[1]
keys = [("rm_a", "RM_A score\n(labeller)", ORANGE), ("rm_b", "RM_B score\n(held out)", BLUE), ("length", "answer\nlength", GRAY)]
names = {"rm_a": "RM_A score (the labeller)", "rm_b": "RM_B score (held-out RM)", "length": "answer length (tokens)"}
sd = {k: before[k].float().std().item() for k, _, _ in keys}      # per-answer spread under π_0
ax.bar([lab for _, lab, _ in keys], [online_dpo.loc[names[k], "change"] / sd[k] for k, _, _ in keys],
       yerr=[2 * online_dpo.loc[names[k], "SE"] / sd[k] for k, _, _ in keys], color=[c for _, _, c in keys], capsize=6, width=0.5)
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("change π_0 → π_1 (in SDs of π_0's answers)"); ax.set_title("fresh samples, held-out questions (± 2 SE)")
ax = axes[2]
for key, color, label in [("RM_A", ORANGE, "RM_A (selects)"), ("RM_B", BLUE, "RM_B (held out)")]:
    gain = (bon_real[key] - bon_real.loc[1, key]) / sd[key.lower().replace("_", "_")]
    ax.errorbar(bon_real.index, gain, yerr=2 * bon_real[f"{key} SE"] / sd[key.lower()], color=color, marker="o", capsize=3, label=label)
ax.axhline(0, color=GRAY, lw=0.8)
ax.set_xscale("log", base=2); ax.set_xticks([1, 2, 4, 8], ["1", "2", "4", "8"])
ax.set_xlabel("n"); ax.set_ylabel("gain over n = 1 (in SDs)"); ax.set_title("best-of-n with RM_A on π_0 (± 2 SE of each mean)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

### Reference explanation

### Reference answer and mechanism

Numbers below are ranges over four reference runs of this notebook. Reward-model training is not
bit-reproducible on the GPU, so the scores differ from run to run, but the pattern was the same in
every run.

(a) **Yes, by far.** `RM_A`'s mean score on fresh held-out samples rose by 1.0–2.1 points, 15–18
standard errors, while the null comparison in Exercise 10 stayed within one SE of 0. The
implicit-reward accuracy on held-out `RM_A`-labelled pairs rose from 50% to 87–96%. DPO learned what its labeller prefers, and that preference transfers to
questions it never trained on. As an optimizer, one round of online DPO works.

(b) **Yes, and that is less reassuring than it sounds.** `RM_B` also rose, by 12–17 SE, though
less in units of its own spread than `RM_A` (middle panel). But the two reward models were trained on
the same kind of data, and their scores agree with each other (r ≈ +0.5 on `π_0`'s samples) much more
than either agrees with the human labels (54–56%). Agreement between two proxies trained on the same
distribution is evidence that they share features, not that those features are quality.

(c) **Yes, a lot.** Answers grew from about 52 to 76–83 tokens (45–58% longer), and the
fraction that hit the 96-token limit rose from about 1% to 15–25%, although every prompt asks for two
or three sentences. On `π_0`'s samples both scores correlate with length (r between +0.36 and +0.60). The
simplest reading is that much of what DPO learned from `RM_A` is *write longer answers*, and `RM_B`
rewards the same thing. The training curve shows the usual DPO pattern on top: the rejected answers'
implicit reward fell much more than the chosen answers' rose.

(d) **No.** Nothing here measures helpfulness: labeller and evaluator are both about 55% accurate,
share a length preference, and no human or independent judge looked at the answers. The examples
printed above contain the same kind of factual errors before and after. A quality claim would need
human or strong-judge evaluation with length controlled.

**Best-of-n** on `π_0` (right panel) shows Part A's pattern on the real model: `RM_A`'s score of the
selected answer rises with `n` by construction, and `RM_B`'s rises too, by less. Eight samples are far
too few to see whether `RM_B` would eventually turn down, as the true reward did in Part A.

Two limits of this run: the DPO loss fell to about 0.1 in the second pass, so the 100–125 pairs are
nearly memorized; and it is one round. Iterative DPO in practice runs several rounds over many
thousands of prompts with a far stronger labeller, and still has to watch length. Stage S7 studies
exactly this failure: a policy that raises a proxy's score through a feature such as length.

**Saved reference preview — rerun the preceding plot cell for current inputs.**

![Online DPO on Qwen3: training curve, before/after change in both reward models, and best-of-n](../figures/chapter-13/study-path-s6_combinations-06.png)

This fixed image comes from the Spark GPU reference run; your own run updates the plot above.

In [ ]:
print(f"total notebook time so far: {(time.time() - T_START) / 60:.1f} min")
del policy, ref, rm_a, rm_b
Q.free()

---

## What you built and measured

| Part | You wrote | What it showed |
|---|---|---|
| A · best-of-n, RSFT | best-of-n selection, the exact best-of-n distribution, the SFT loss on selections | selecting with a flawed reward model raises its score monotonically while the true reward peaks (here at n = 4) and falls; SFT on the selections converges to the best-of-n distribution, and k rounds with a fixed reward model equal best-of-n^k (exact identity) |
| B · offline vs online DPO | hard and Bradley–Terry labelling, the KL-regularized optimum | `J(π*) − J(π) = β·KL(π ‖ π*)`; online DPO with Bradley–Terry labels converges to π* (KL ≈ 0.01); offline DPO can't move answers outside its dataset (logits bit-identical); hard labels overshoot π* |
| C · RM + verifier | group z-normalisation, summed and gated rewards | z-scores keep order and drop gap size; a sum with w = 1 ends on the wrong answer the RM likes, the gate ends on the best correct answer |
| D · SFT → DPO → RLVR | the KL-anchored RLVR loss | SFT sets the format, DPO the style (including a wrong-but-preferred answer), RLVR the correctness; the anchor keeps DPO's ratios among correct answers (matches the closed form) |
| E · Qwen3 online DPO | best-vs-worst pairs with a margin, a paired standard error | one round on 100–125 RM-labelled pairs: both reward models' scores rose by more than 10 SE, but answers also got about 50% longer and both scores correlate with length (r ≈ +0.4 to +0.6 on π_0's samples); a mechanics result, not a quality result |

**Evidence.** Parts A–D are exact or near-exact computations on tabular policies. The identities
(best-of-n formula, compounding of selection rounds, the KL-optimum gap, zero gradient for answers
absent from offline pairs, ratio preservation under a KL anchor) hold for any table. The converged
policies are specific to the hand-written numbers, and a table can't show how a change on one
prompt spreads to others through shared weights. Part E establishes that the online loop runs end to
end on Qwen3-0.6B (sample, label with a reward model, DPO, re-evaluate), within 22 GiB of GPU
memory, and it reports the changes in reward-model scores with paired standard errors. It does
**not** establish that the policy became more helpful: the labeller is about 55% accurate on
held-out human labels, the second reward model is trained on the same kind of data and can share
its biases, and there is no human or independent judge evaluation. One round with a few hundred
pairs is a mechanics check.

**Next stage.** [S7 · When optimization goes wrong](s7_failure_modes.ipynb) studies the failure
this notebook kept running into: a policy that raises a proxy's score without becoming better
(reward hacking, length and entropy effects), on the same toy-then-Qwen3 pattern.